# Fatigue modeling

Ordinal models for `fatigue_num` (0–5) with participant-level held-out test, GroupKFold CV, and Optuna tuning. Core logic lives in `src/modeling/`.

Tuning and CV use **train/val participants only**; held-out test participants never appear in Optuna or CV folds.

In [1]:
%pip install -q -r ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path

_src = Path('../../src').resolve()
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# Ensure local src edits are picked up when re-running this cell.
for _mod in [k for k in list(sys.modules) if k == 'modeling' or k.startswith('modeling.')]:
    del sys.modules[_mod]

import pandas as pd
from modeling.baselines import (
    run_all_baseline_benchmarks,
    summarize_baseline_metrics,
)
from modeling.config import (
    DATA_PATH,
    N_CV_FOLDS,
    OPTUNA_TRIALS,
)
from modeling.data import load_fatigue_data, prepare_splits, split_summary_table
from modeling.registry import ORDINAL_MODELS
from modeling.runner import tune_and_benchmark_model


## 1. Load data and split

Participants are held out with a **stratified split** on per-participant **mean fatigue** (`fatigue_num` averaged over each participant's days; `prepare_splits(..., stratify=True)`) so train/val and test have similar average fatigue levels.

“We randomly assign whole participants to train/val or test, but we do it in a way that both groups contain a similar mix of people with low, medium, and high average fatigue — not just a random 8 people who might all happen to be high-average-fatigue reporters.”


In [3]:
df = load_fatigue_data('../../' + DATA_PATH)
bundle = prepare_splits(df)

print(f"Rows: {len(df):,}  Participants: {df['id'].nunique()}")
display(split_summary_table(bundle))
print('Test participant ids:', sorted(bundle.test_ids))


Rows: 3,331  Participants: 42


,split,participants,rows,mean_fatigue
0,train_val,34,2659,2.462204
1,test,8,672,2.653274


Test participant ids: [np.int64(7), np.int64(14), np.int64(24), np.int64(38), np.int64(40), np.int64(41), np.int64(46), np.int64(50)]


Re-run the **init accumulators** cell below before a fresh partial run to clear prior tuned-model results.


In [4]:
# Re-run this cell to clear accumulated model results before a fresh partial run.
ordinal_results = []
history_ordinal_results = []

ordinal_best_params = {}
history_best_params = {}


## 2. Baseline benchmarks

Simple predictors evaluated with the same GroupKFold CV and held-out test protocol as the tuned models. Includes persistence baselines **`lag1_fatigue`** and **`expanding_mean`**.


In [5]:
ordinal_baseline_results = run_all_baseline_benchmarks(bundle, task='ordinal', n_splits=N_CV_FOLDS)

ordinal_baseline_summary = summarize_baseline_metrics(ordinal_baseline_results, task='ordinal')

print('Ordinal baselines (test metrics)')
display(ordinal_baseline_summary[[c for c in ordinal_baseline_summary.columns if c.startswith('test_')]])


Ordinal baselines (test metrics)


,test_mae,test_rmse,test_r2,test_qwk
model,,,,
global_mean,1.406250,1.640721,-0.188402,0.000000
global_mode,1.156250,1.544479,-0.053072,0.000000
lag1_fatigue,0.950893,1.424175,0.104593,0.549449
expanding_mean,1.025298,1.336863,0.211017,0.422289


MAE: Mean Absolute Error;

RMSE: Root Mean Squared Error, measures the variation in residual/error

R2: how much variability is explained by the model

QWK: Quadratic Weighted Kappa. QWK measures the agreement between two raters—such as an AI and a human—on an ordered scale. It is designed to adjust for chance agreements and heavily penalize larger scoring discrepancies over minor ones.

## 3. Train/Tune models

### Ordinal Regression

Continuous loss on `fatigue_num`, then round and clip to [0, 5].

#### `linear_regression`


In [6]:
_name = 'linear_regression'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] linear_regression  test_mae=1.3452


#### `ordinal_rf`


In [7]:
_name = 'ordinal_rf'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordinal_rf  test_mae=1.3289


#### `catboost_regressor`


In [8]:
_name = 'catboost_regressor'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] catboost_regressor  test_mae=1.1994


### Ordinal Classification

Ordered likelihood or threshold structure on `fatigue_num` 0–5. Evaluated with the same MAE / QWK metrics as regression models.

#### `ordered_logistic`


In [9]:
_name = 'ordered_logistic'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordered_logistic  test_mae=1.3110


#### `ordinal_forest`


In [10]:
_name = 'ordinal_forest'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordinal_forest  test_mae=1.2173


#### `mixed_effects`


In [11]:
_name = 'mixed_effects'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] mixed_effects  test_mae=1.4554


#### `catboost_ordinal`


In [12]:
_name = 'catboost_ordinal'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] catboost_ordinal  test_mae=1.1414


### History (feature ablation)

Same seven ordinal models as above, with **seven extra leakage-safe history columns** appended to the daily feature matrix. History features use fixed defaults from `prepare_splits` (`ewma_alpha=0.3`, `rolling_window=3`); first-day NaNs in history columns are imputed with the train/val median.

**History features** (7 cols):
- fatigue lag1: Yesterday's fatigue score
- fatigue EWMA: Exponentially weighted average of past fatigue; recent days count more
- fatigue expanding mean: Average fatigue on all earlier days for this person
- fatigue delta lag1: Change in fatigue, the worsening/improving trend
- activity_logsum_roll3_mean: Rolling mean of prior days' sum of log1p(lightly) + log1p(moderately) + log1p(very)
- calories_sum_roll3_mean: Recent typical daily calories burned
- very_roll3_mean: Recent typical "very active" minutes


#### Ordinal Regression (history)

Continuous loss on `fatigue_num`, then round and clip to [0, 5].


##### `linear_regression` (history)


In [13]:
_name = 'linear_regression'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    feature_set='history',
    display_name=f'{_name}_history',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[f'{_name}_history'] = _params
print(f'[ok] {_name}_history  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] linear_regression_history  test_mae=0.8899


##### `ordinal_rf` (history)


In [14]:
_name = 'ordinal_rf'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    feature_set='history',
    display_name=f'{_name}_history',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[f'{_name}_history'] = _params
print(f'[ok] {_name}_history  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordinal_rf_history  test_mae=0.8929


##### `catboost_regressor` (history)


In [15]:
_name = 'catboost_regressor'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    feature_set='history',
    display_name=f'{_name}_history',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[f'{_name}_history'] = _params
print(f'[ok] {_name}_history  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] catboost_regressor_history  test_mae=0.9107


#### Ordinal Classification (history)


##### `ordered_logistic` (history)


In [16]:
_name = 'ordered_logistic'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    feature_set='history',
    display_name=f'{_name}_history',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[f'{_name}_history'] = _params
print(f'[ok] {_name}_history  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordered_logistic_history  test_mae=0.8929


##### `ordinal_forest` (history)


In [17]:
_name = 'ordinal_forest'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    feature_set='history',
    display_name=f'{_name}_history',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[f'{_name}_history'] = _params
print(f'[ok] {_name}_history  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordinal_forest_history  test_mae=0.8988


##### `mixed_effects` (history)


In [18]:
_name = 'mixed_effects'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    feature_set='history',
    display_name=f'{_name}_history',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[f'{_name}_history'] = _params
print(f'[ok] {_name}_history  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] mixed_effects_history  test_mae=0.8720


##### `catboost_ordinal` (history)


In [19]:
_name = 'catboost_ordinal'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    feature_set='history',
    display_name=f'{_name}_history',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[f'{_name}_history'] = _params
print(f'[ok] {_name}_history  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] catboost_ordinal_history  test_mae=0.8884


## 4. Results summary

Includes baselines plus only models whose §3 cells were executed. Ordinal CV summary shows **`cv_mae_std` only** (fold stability for MAE).

In [20]:
def collect_summaries(results, task='ordinal'):
    cv_rows, test_rows = [], []
    if task == 'ordinal':
        metric_cols = ['mae', 'rmse', 'r2', 'qwk']
    else:
        metric_cols = ['weighted_f1', 'macro_f1', 'accuracy']
    for result in results:
        cv_mean = result['cv_summary'].loc['mean', metric_cols]
        cv_std = result['cv_summary'].loc['std', metric_cols]
        cv_row = {
            'model': result['name'],
            'best_params': str(result.get('best_params', {})),
        }
        test_row = {
            'model': result['name'],
            'best_params': str(result.get('best_params', {})),
        }
        for col in metric_cols:
            cv_row[f'cv_{col}'] = cv_mean[col]
            test_row[f'test_{col}'] = result['test_metrics'][col]
        if task == 'ordinal':
            cv_row['cv_mae_std'] = cv_std['mae']
        cv_rows.append(cv_row)
        test_rows.append(test_row)
    return pd.DataFrame(cv_rows).set_index('model'), pd.DataFrame(test_rows).set_index('model')

ordinal_results = globals().get('ordinal_results', [])
history_ordinal_results = globals().get('history_ordinal_results', [])
ordinal_best_params = globals().get('ordinal_best_params', {})
history_best_params = globals().get('history_best_params', {})

ran_ordinal = sorted(set(ordinal_best_params) | set(history_best_params))
print(f'Ran {len(ran_ordinal)} tuned ordinal models: {ran_ordinal}')

all_ordinal_results = ordinal_baseline_results + ordinal_results + history_ordinal_results
ordinal_cv_summary, ordinal_test_summary = collect_summaries(all_ordinal_results, task='ordinal')

print('Ordinal CV summary (baselines first)')
display(ordinal_cv_summary)
print('Ordinal held-out test summary')
display(ordinal_test_summary)


Ran 14 tuned ordinal models: ['catboost_ordinal', 'catboost_ordinal_history', 'catboost_regressor', 'catboost_regressor_history', 'linear_regression', 'linear_regression_history', 'mixed_effects', 'mixed_effects_history', 'ordered_logistic', 'ordered_logistic_history', 'ordinal_forest', 'ordinal_forest_history', 'ordinal_rf', 'ordinal_rf_history']
Ordinal CV summary (baselines first)


,best_params,cv_mae,cv_rmse,cv_r2,cv_qwk,cv_mae_std
model,,,,,,
global_mean,{},1.348516,1.606173,-0.297646,0.000000,0.153930
global_mode,{},1.216046,1.554387,-0.200378,0.000000,0.243084
lag1_fatigue,{},0.824015,1.315205,0.096947,0.546287,0.193110
expanding_mean,{},0.867974,1.198179,0.280799,0.495990,0.127344
linear_regression,{'alpha': 8.799396078570537},1.464940,1.730648,-0.517784,-0.003573,0.239878
ordinal_rf,"{'n_estimators': 119, 'max_depth': 6, 'min_sam...",1.290071,1.596975,-0.270443,0.066289,0.170864
catboost_regressor,"{'iterations': 213, 'depth': 9, 'learning_rate...",1.233081,1.507669,-0.134556,0.112615,0.130442
ordered_logistic,{'alpha': 9.970600386746954},1.546550,1.825137,-0.715481,-0.013961,0.212164
ordinal_forest,"{'n_estimators': 227, 'max_depth': 9, 'min_sam...",1.227097,1.492878,-0.111456,0.125225,0.140734


Ordinal held-out test summary


,best_params,test_mae,test_rmse,test_r2,test_qwk
model,,,,,
global_mean,{},1.406250,1.640721,-0.188402,0.000000
global_mode,{},1.156250,1.544479,-0.053072,0.000000
lag1_fatigue,{},0.950893,1.424175,0.104593,0.549449
expanding_mean,{},1.025298,1.336863,0.211017,0.422289
linear_regression,{'alpha': 8.799396078570537},1.345238,1.622021,-0.161467,-0.004347
ordinal_rf,"{'n_estimators': 119, 'max_depth': 6, 'min_sam...",1.328869,1.640721,-0.188402,0.068100
catboost_regressor,"{'iterations': 213, 'depth': 9, 'learning_rate...",1.199405,1.540138,-0.047160,0.079181
ordered_logistic,{'alpha': 9.970600386746954},1.311012,1.644345,-0.193657,-0.008234
ordinal_forest,"{'n_estimators': 227, 'max_depth': 9, 'min_sam...",1.217262,1.533359,-0.037963,0.064540


In [21]:
_pair_rows = []
for _base in ORDINAL_MODELS:
    _hist = f'{_base}_history'
    if _base not in ordinal_test_summary.index or _hist not in ordinal_test_summary.index:
        continue
    _tabular_mae = ordinal_test_summary.loc[_base, 'test_mae']
    _history_mae = ordinal_test_summary.loc[_hist, 'test_mae']
    _pair_rows.append({
        'model': _base,
        'test_mae_tabular': _tabular_mae,
        'test_mae_history': _history_mae,
        'delta_mae': _history_mae - _tabular_mae,
    })

if _pair_rows:
    history_ablation_summary = (
        pd.DataFrame(_pair_rows)
        .set_index('model')
        .sort_values('delta_mae')
    )
    print('Tabular vs history paired comparison (delta_mae = history - tabular; negative = history helps)')
    display(history_ablation_summary)
else:
    print('No paired tabular/history models found — run both §3 blocks first.')


Tabular vs history paired comparison (delta_mae = history - tabular; negative = history helps)


,test_mae_tabular,test_mae_history,delta_mae
model,,,
mixed_effects,1.455357,0.872024,-0.583333
linear_regression,1.345238,0.889881,-0.455357
ordinal_rf,1.328869,0.892857,-0.436012
ordered_logistic,1.311012,0.892857,-0.418155
ordinal_forest,1.217262,0.898810,-0.318452
catboost_regressor,1.199405,0.910714,-0.288690
catboost_ordinal,1.141369,0.888393,-0.252976
